# DataInsight | Tabular: ✏️ <tên đề tài>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/tabular.ipynb)

| | |
|---|---|
| **Môn học** | Nền tảng lập trình cho phân tích và trực quan dữ liệu (CO5177), Học kỳ 261, 2026–2027 |
| **Giảng viên** | Lê Thành Sách, Khoa Khoa học và Kỹ thuật Máy tính, ĐH Bách Khoa – ĐHQG-HCM |
| **Nhóm** | DataInsight: Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Người thực hiện** | Trần Lê Hoàng Long (2570443) |
| **Dataset** | ✏️ [<tên dataset>](<link>), <nguồn>, <license> |
| **Bài toán** | ✏️ Phân loại nhị phân / nhiều lớp cột `<target>` từ `<n>` đặc trưng |

**Mô tả bài toán.** ✏️ Một đoạn: dự đoán gì, từ dữ liệu đầu vào nào, vì sao quan trọng, và metric nào dùng để chọn mô hình tốt nhất.

**Mục lục**
1. [Cài đặt](#s1)
2. [Tải dữ liệu](#s2)
3. [Mô tả tập dữ liệu](#s3)
4. [Kiểm tra chất lượng dữ liệu](#s4)
5. [Phân tích khám phá (EDA)](#s5)
6. [Chuẩn bị dữ liệu](#s6)
7. [Mô hình hóa](#s7)
8. [Đánh giá trên tập test](#s8)
9. [Dự đoán trên dữ liệu mới](#s9)
10. [Tổng kết](#s10)

<a id="s1"></a>
## 1. Cài đặt

### 1.1 Cài thư viện còn thiếu

Google Colab đã có sẵn phần lớn thư viện. Cell dưới chỉ cài gói nào Python không tìm thấy, nên Run all trên Colab vẫn nhanh và vẫn chạy được trên môi trường local mới.

In [ ]:
import importlib.util
import subprocess
import sys

# tên module khi import -> gói pip tương ứng. Thêm một dòng cho mỗi thư viện dùng thêm.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
}

# find_spec trả về None khi module chưa được cài; chưa import gì nên không cần restart.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Đã có đủ thư viện cần thiết.")

### 1.2 Import thư viện

Nhóm theo vai trò. Mỗi import của scikit-learn dùng cho đúng một bước, ghi ở comment.

In [ ]:
import itertools
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np                         # mảng, tính toán vector hóa
import pandas as pd                        # bảng: đọc, làm sạch, gom nhóm, tách X / y
import plotly.express as px                # biểu đồ Plotly nhanh từ DataFrame
import plotly.graph_objects as go          # biểu đồ Plotly tùy chỉnh chi tiết
import plotly.io as pio                    # template Plotly dùng chung
from plotly.subplots import make_subplots  # nhiều panel trong một hình

import sklearn
from sklearn.base import clone                                  # bản sao chưa fit của một pipeline
from sklearn.compose import ColumnTransformer                   # tiền xử lý khác nhau cho từng nhóm cột
from sklearn.dummy import DummyClassifier                       # baseline bỏ qua đặc trưng
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer                        # điền missing bằng thống kê của train
from sklearn.inspection import permutation_importance           # độ quan trọng đặc trưng, dùng được cho mọi mô hình
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline                           # nối tiền xử lý + mô hình
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)  # chỉ ẩn cảnh báo deprecation của thư viện; giữ cảnh báo thật
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

### 1.3 Cấu hình

**Mọi thứ phụ thuộc vào dataset chỉ được đặt ở đây.** Phần còn lại của notebook đọc các tên này, nên đổi dataset chỉ cần sửa cell này.

In [ ]:
# ---- Tái lập kết quả ---------------------------------------------------------------------------
SEED = 42                     # một seed cho mọi bước ngẫu nhiên (split, mô hình, CV) -> Run all ra cùng số liệu
np.random.seed(SEED)

# ---- Nguồn dữ liệu (mục 2) ---------------------------------------------------------------------
DATA_DIR = Path("data")       # cache local, đã gitignore
DATA_SOURCE = {
    "kind": "TODO",           # "url" | "gdrive" | "kaggle"
    "location": "TODO",       # URL trực tiếp | file id Google Drive | handle Kaggle "owner/dataset"
    "filename": "TODO.csv",   # file dữ liệu cần đọc sau khi tải (tìm đệ quy)
    # "archive": "TODO.zip",  # bỏ comment khi link url/gdrive là file .zip
}
READ_KWARGS = {}              # tham số thêm cho pandas, vd {"sep": ";", "encoding": "latin-1"}

# ---- Định nghĩa bài toán -----------------------------------------------------------------------
TARGET = "TODO"               # cột nhãn
ID_COLS = []                  # cột định danh (customer_id, ...): giữ để tra cứu, không dùng làm đặc trưng
DROP_COLS = []                # cột loại bỏ có chủ đích (leakage, trùng lặp, ...); giải thích từng cột ở mục 6.1
FORCE_CATEGORICAL = []        # mã trông giống số nhưng thực chất là phân loại (mã bưu chính, mã sản phẩm, ...)
NA_TOKENS = ["?", "NA", "N/A", "n/a", "null", "NULL", ""]  # các chuỗi mang nghĩa "thiếu" trong file

# ---- Quy tắc miền (mục 4.4 và 6.1) -------------------------------------------------------------
VALID_RANGES = {}             # TODO: {"age": (18, 100), "income": (0, None)}; None = không giới hạn. Theo kiến thức miền, không theo thống kê
SPELLING_MAPS = {}            # TODO: {"gender": {"m": "Male", "male": "Male", "f": "Female"}}; key đã strip + viết thường
TO_NUMERIC = []               # TODO: cột số bị lưu dạng chữ ("1,234"), phát hiện ở mục 4.3

# ---- Chia dữ liệu và tiền xử lý ----------------------------------------------------------------
TEST_SIZE, VAL_SIZE = 0.15, 0.15   # 70% train / 15% validation / 15% test
MIN_CATEGORY_FREQ = 0.01           # category hiếm hơn 1% của train được gộp vào một cột "infrequent"

# Dừng Run all sớm với thông báo rõ ràng thay vì lỗi ở 20 cell sau.
for _name, _value in {"DATA_SOURCE": DATA_SOURCE, "TARGET": TARGET}.items():
    assert "TODO" not in str(_value), f"Hãy điền {_name} ở mục 1.3 trước khi chạy notebook."

### 1.4 Phong cách biểu đồ và hàm dùng chung

Mọi hình dùng **Plotly** (tương tác: rê chuột xem giá trị, zoom được). Một template chung giữ màu và font giống nhau ở cả ba notebook của nhóm; bảng màu trùng với notebook Time series.

Quy tắc cho mọi hình:
- Tạo bằng `plotly.express` / `plotly.graph_objects`, rồi gọi `show_fig(fig, title)`. Hàm tự đánh số hình (`Figure 1.`, `Figure 2.`, …), nên thêm cell không làm lệch số thứ tự.
- Tiêu đề nêu **kết luận** ("Class 1 is only 12% of samples"), không nêu loại biểu đồ. Chữ trên hình viết tiếng Anh, thống nhất với notebook Time series.
- Hai trục có nhãn và đơn vị; sau mỗi hình là một cell nhận xét có số liệu.

In [ ]:
# Màu theo vai trò, thứ tự cố định: chuỗi phân loại dùng BLUE, ORANGE, AQUA, YELLOW, rồi RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # số đếm, cường độ
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # tương quan (-1..1)

# Một template Plotly có tên = một chỗ duy nhất để đổi giao diện mọi biểu đồ.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

_figure_number = itertools.count(1)  # bộ đếm toàn cục: Run all đánh số hình 1, 2, 3, ... theo thứ tự


def show_fig(fig, title, height=420, **layout):
    """Đánh số hình, gắn tiêu đề dạng kết luận rồi hiển thị."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram tính sẵn bằng NumPy: notebook chỉ lưu ~40 cột thay vì mọi điểm dữ liệu."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def grid_position(i, ncols):
    """(hàng, cột) của panel thứ i trong lưới make_subplots, điền từ trái sang phải."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Tải dữ liệu

### 2.1 Tải về

Dữ liệu **không bao giờ upload tay**: notebook tự tải từ nguồn công khai và lưu cache vào `data/` (đã gitignore), nên giảng viên chỉ cần bấm Run all. `download_dataset` hỗ trợ ba loại nguồn:

| `kind` | `location` | Thư viện | Ghi chú |
|---|---|---|---|
| `"url"` | link tải trực tiếp | `urllib` (thư viện chuẩn) | GitHub raw, UCI, Hugging Face `resolve/...`, … |
| `"gdrive"` | **file id** Google Drive (phần sau `/d/` trong link chia sẻ) | `gdown` | Chia sẻ dạng *Anyone with the link → Viewer* |
| `"kaggle"` | handle `owner/dataset-name` | `kagglehub` | Dataset công khai |

Nếu link là file `.zip`, đặt thêm `"archive"`; file nén được giải nén và `filename` được tìm bên trong.

In [ ]:
def download_dataset(source, data_dir=DATA_DIR):
    """Tải một nguồn dữ liệu (bỏ qua nếu đã có cache) và trả về đường dẫn của `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub có cache riêng
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: không bao giờ tải hai lần
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Loại nguồn không hợp lệ {kind!r}: dùng 'url', 'gdrive' hoặc 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: file có thể nằm trong thư mục con của file nén
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"Không tìm thấy {filename!r} trong {root}. Các file có: {found}")
    print(f"File dữ liệu: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Đọc file dữ liệu bằng hàm đọc pandas khớp với phần mở rộng."""
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"Chưa có hàm đọc cho {suffix!r}; thêm vào read_table.")

### 2.2 Đọc bảng dữ liệu

In [ ]:
data_path = download_dataset(DATA_SOURCE)
raw = read_table(data_path, na_values=NA_TOKENS, **READ_KWARGS)
raw.columns = raw.columns.str.strip()  # tên cột thừa khoảng trắng sẽ làm lỗi khi truy cập cột về sau
assert TARGET in raw.columns, f"Không tìm thấy TARGET {TARGET!r}. Các cột: {list(raw.columns)}"

print(f"Kích thước: {raw.shape[0]:,} dòng x {raw.shape[1]} cột")
raw.head()

### 2.3 Dataset có đạt yêu cầu của đề bài?

Đề bài §4.2 yêu cầu dữ liệu dạng bảng có ≥ 2.000 mẫu, ≥ 10 cột, có missing values, có cột categorical và numerical. Bảng dưới là bằng chứng cho báo cáo.

In [ ]:
_features = [c for c in raw.columns if c not in [TARGET, *ID_COLS]]
_n_numeric = len(raw[_features].select_dtypes("number").columns)
requirements = pd.DataFrame([
    ("Ít nhất 2.000 mẫu", f"{len(raw):,} dòng", len(raw) >= 2000),
    ("Ít nhất 10 cột", f"{raw.shape[1]} cột", raw.shape[1] >= 10),
    ("Có missing values", f"{int(raw.isna().sum().sum()):,} ô thiếu", bool(raw.isna().any().any())),
    ("Có cột categorical", f"{len(_features) - _n_numeric} cột không phải số", len(_features) > _n_numeric),
    ("Có cột numerical", f"{_n_numeric} cột số", _n_numeric > 0),
], columns=["Yêu cầu (đề bài §4.2)", "Dataset này", "Đạt"])
requirements

<a id="s3"></a>
## 3. Mô tả tập dữ liệu

✏️ Nguồn gốc dữ liệu (ai thu thập, khi nào, bằng cách nào), một dòng biểu diễn điều gì, license.

| Cột | Kiểu | Ý nghĩa | Vai trò |
|---|---|---|---|
| ✏️ `col_a` | numeric | … | đặc trưng |
| ✏️ `col_b` | categorical | … | đặc trưng |
| ✏️ `target` | categorical | … | **target** |

In [ ]:
raw.info()  # kiểu dữ liệu từng cột: object/str = chữ, int64/float64 = số; cột số có NaN sẽ thành float64

### 3.1 Vai trò các cột

Các cột được chia tự động theo kiểu dữ liệu; `FORCE_CATEGORICAL` ghi đè cho các mã trông giống số. Hai danh sách này điều khiển các biểu đồ ở mục 4–5 và tiền xử lý ở mục 6.

In [ ]:
def column_roles(df):
    """Chia các cột đặc trưng thành danh sách numeric và categorical, theo cấu hình."""
    features = [c for c in df.columns if c not in {TARGET, *ID_COLS, *DROP_COLS}]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(df[c]) and c not in FORCE_CATEGORICAL]
    categorical = [c for c in features if c not in numeric]
    return features, numeric, categorical


FEATURES, NUM_COLS, CAT_COLS = column_roles(raw)
print(f"{len(NUM_COLS)} cột numeric:", NUM_COLS)
print(f"{len(CAT_COLS)} cột categorical:", CAT_COLS)

In [ ]:
raw[NUM_COLS].describe().T  # count < số dòng => có missing; min/max lộ ra giá trị vô lý

In [ ]:
cat_summary = pd.DataFrame({
    "số giá trị khác nhau": raw[CAT_COLS].nunique(),
    "giá trị phổ biến nhất": raw[CAT_COLS].mode().iloc[0] if CAT_COLS else [],
    "tỷ lệ giá trị phổ biến nhất": [raw[c].value_counts(normalize=True).iloc[0] for c in CAT_COLS],
    "số ô thiếu": raw[CAT_COLS].isna().sum(),
})
cat_summary  # nhiều giá trị khác nhau = cardinality cao -> one-hot sẽ tạo nhiều cột (xem MIN_CATEGORY_FREQ)

**Nhận xét.** ✏️ Số dòng/cột, cột nào numeric hay categorical, cột có kiểu dữ liệu bất ngờ, cột có cardinality cao.

<a id="s4"></a>
## 4. Kiểm tra chất lượng dữ liệu

Mục này **chỉ chẩn đoán**; mọi cách xử lý được áp dụng ở mục 6.1 để các quy tắc làm sạch nằm trong một hàm. Theo bài giảng, bốn loại lỗi cần bốn cách xử lý khác nhau: missing values, giá trị vô lý, dòng trùng và chính tả không thống nhất. Mất cân bằng lớp và outlier cũng được kiểm tra vì chúng thay đổi lựa chọn mô hình.

### 4.1 Missing values

Hai cách đếm: `NaN` thật, và missing **ẩn**: các chuỗi như `" ?"` mà `na_values` không bắt được vì có khoảng trắng.

In [ ]:
text_cols = raw.columns.difference(raw.select_dtypes("number").columns)
hidden = {c: int(raw[c].dropna().astype(str).str.strip().isin(NA_TOKENS).sum()) for c in text_cols}

missing = pd.DataFrame({"NaN": raw.isna().sum(), "chuỗi ẩn": pd.Series(hidden)}).fillna(0).astype(int)
missing["tổng"] = missing["NaN"] + missing["chuỗi ẩn"]
missing["% số dòng"] = missing["tổng"] / len(raw) * 100
missing = missing[missing["tổng"] > 0].sort_values("tổng", ascending=False)

if missing.empty:
    print("Không có missing values.")
else:
    fig = go.Figure(go.Bar(x=missing.index, y=missing["% số dòng"], marker_color=BLUE,
                           text=[f"{p:.1f}%" for p in missing["% số dòng"]], textposition="outside",
                           hovertemplate="%{x}: %{y:.2f}% of rows<extra></extra>"))
    show_fig(fig, f"{len(missing)} columns contain missing values (max {missing['% số dòng'].max():.1f}% of rows)",
             xaxis_title="Column", yaxis_title="% of rows missing")
missing

**Missing có liên quan đến target không?** Nếu phân bố lớp khác nhau giữa các dòng thiếu và không thiếu một giá trị, thì bản thân việc thiếu đã mang thông tin (dữ liệu không thiếu hoàn toàn ngẫu nhiên).

In [ ]:
target_share = raw[TARGET].value_counts(normalize=True)
MINORITY = target_share.idxmin()  # lớp quan trọng nhất khi dữ liệu mất cân bằng

rows = []
for c in missing.index:
    is_na = raw[c].isna() | raw[c].astype(str).str.strip().isin(NA_TOKENS)
    if c == TARGET or is_na.all():
        continue
    rows.append({"cột": c,
                 f"% '{MINORITY}' khi thiếu": (raw.loc[is_na, TARGET] == MINORITY).mean() * 100,
                 f"% '{MINORITY}' khi có": (raw.loc[~is_na, TARGET] == MINORITY).mean() * 100})
pd.DataFrame(rows)

**Nhận xét.** ✏️ Cột nào thiếu, thiếu bao nhiêu, có chuỗi ẩn không, missing có liên quan đến target không. Quyết định: điền giá trị (median / most frequent, fit trên train ở mục 6.3), bỏ cột (thiếu > ~40–50%), hay bỏ dòng (chỉ khi rất ít).

### 4.2 Dòng trùng

- **Trùng hoàn toàn** làm một số mẫu bị nhân lên và có thể đưa cùng một mẫu vào cả train lẫn test (leakage).
- **Cùng đặc trưng, khác nhãn** là các mẫu mâu thuẫn: không mô hình nào đoán đúng hết, nên chúng giới hạn độ chính xác tối đa.

In [ ]:
non_id = [c for c in raw.columns if c not in ID_COLS]
feature_like = [c for c in non_id if c != TARGET]

dup_table = pd.DataFrame({
    "kiểm tra": ["Dòng trùng hoàn toàn", "Trùng khi bỏ qua cột ID", "Giá trị ID lặp lại",
                 "Cùng đặc trưng, khác nhãn (số nhóm)"],
    "số lượng": [
        int(raw.duplicated().sum()),
        int(raw.duplicated(subset=non_id).sum()),
        int(raw.duplicated(subset=ID_COLS).sum()) if ID_COLS else 0,
        int((raw.groupby(feature_like, dropna=False)[TARGET].nunique() > 1).sum()),
    ],
})
dup_table

**Nhận xét.** ✏️ Có bao nhiêu dòng trùng; là bản ghi lặp thật hay do cách thu thập; quyết định (bỏ dòng trùng hoàn toàn trước khi chia tập).

### 4.3 Chính tả không thống nhất và số lưu dạng chữ

`"HCM"`, `" HCM"` và `"hcm"` là ba category khác nhau với pandas. Bảng so sánh số giá trị khác nhau trước và sau `strip().lower()`; nếu giảm nghĩa là có nhiều biến thể của cùng một giá trị. Bảng cũng đánh dấu các cột chữ mà phần lớn giá trị là số.

In [ ]:
rows, variant_examples = [], {}
for c in CAT_COLS:
    s = raw[c].dropna().astype(str)
    norm = s.str.strip().str.lower()
    rows.append({"cột": c,
                 "số giá trị (gốc)": s.nunique(),
                 "số giá trị (strip + lower)": norm.nunique(),
                 "giá trị có khoảng trắng ở đầu/cuối": int((s != s.str.strip()).sum()),
                 "tỷ lệ đọc được thành số": pd.to_numeric(s.str.replace(",", ""), errors="coerce").notna().mean()})
    groups = s.groupby(norm).unique()
    variants = groups[groups.map(len) > 1]
    if len(variants):
        variant_examples[c] = variants.head(5).map(list).to_dict()

display(pd.DataFrame(rows))
for c, examples in variant_examples.items():
    print(f"{c}: {examples}")

**Nhận xét.** ✏️ Cột có biến thể chính tả (→ thêm vào `SPELLING_MAPS`) và cột chữ nên là số (→ thêm vào `TO_NUMERIC`).

### 4.4 Giá trị vô lý

Giá trị không thể đúng (tuổi 200, thu nhập âm) là **lỗi**, khác với outlier có thể là thật. Quy tắc lấy từ kiến thức miền và được viết trong `VALID_RANGES`.

In [ ]:
rows = []
for c, (low, high) in VALID_RANGES.items():
    values = pd.to_numeric(raw[c], errors="coerce")
    too_low = int((values < low).sum()) if low is not None else 0
    too_high = int((values > high).sum()) if high is not None else 0
    rows.append({"cột": c, "quy tắc": f"[{low}, {high}]", "dưới ngưỡng": too_low, "trên ngưỡng": too_high,
                 "min quan sát": values.min(), "max quan sát": values.max()})
pd.DataFrame(rows) if rows else print("VALID_RANGES đang trống: thêm quy tắc miền ở mục 1.3.")

**Nhận xét.** ✏️ Quy tắc nào bị vi phạm và bao nhiêu lần. Quyết định: bỏ dòng (như trong bài giảng) hoặc đặt thành NaN rồi điền giá trị.

### 4.5 Outliers (quy tắc IQR)

Một giá trị bị đánh dấu khi nằm ngoài $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$ với $IQR = Q_3 - Q_1$. Giá trị bị đánh dấu **không mặc định là lỗi**: thu nhập rất cao vẫn có thể là thật. Boxplot cho thấy chúng xa đến đâu.

In [ ]:
q1, q3 = raw[NUM_COLS].quantile(0.25), raw[NUM_COLS].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = pd.DataFrame({"ngưỡng dưới": lower, "ngưỡng trên": upper,
                         "số dưới ngưỡng": (raw[NUM_COLS] < lower).sum(), "số trên ngưỡng": (raw[NUM_COLS] > upper).sum()})
outliers["% số dòng"] = (outliers["số dưới ngưỡng"] + outliers["số trên ngưỡng"]) / len(raw) * 100
outliers.sort_values("% số dòng", ascending=False)

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:12]  # giữ hình dễ đọc; nếu nhiều cột hơn thì chọn các cột quan trọng nhất
nrows = max(1, -(-len(plot_cols) // ncols))  # chia làm tròn lên
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Box(y=raw[c], name=c, marker_color=BLUE, boxpoints="outliers", showlegend=False), row=r, col=k)
show_fig(fig, "Spread and outliers of the numeric columns (each panel has its own scale)", height=300 * nrows)

**Nhận xét.** ✏️ Cột nào có đuôi dài; giá trị thật hay lỗi? Quyết định: giữ nguyên (mô hình cây không nhạy), cắt ngưỡng, biến đổi log, hay coi là lỗi ở 4.4.

### 4.6 Cân bằng lớp

Accuracy dễ gây hiểu nhầm khi một lớp áp đảo: luôn đoán lớp đa số đã được điểm bằng tỷ lệ của lớp đó. Điều này quyết định metric (F1-macro, balanced accuracy) và có dùng `class_weight="balanced"` hay không.

In [ ]:
counts = raw[TARGET].value_counts(dropna=False)
shares = counts / counts.sum() * 100
fig = go.Figure(go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=BLUE,
                       text=[f"{n:,} ({p:.1f}%)" for n, p in zip(counts, shares)], textposition="outside",
                       hovertemplate="%{x}: %{y:,} rows<extra></extra>"))
imbalance_ratio = counts.max() / counts.min()
show_fig(fig, f"Class balance of '{TARGET}': majority/minority ratio = {imbalance_ratio:.1f}",
         xaxis_title=TARGET, yaxis_title="Rows")
print(f"Accuracy khi luôn đoán lớp đa số: {shares.max():.1f}%")

**Nhận xét.** ✏️ Tỷ lệ các lớp, tỷ số mất cân bằng, hệ quả cho metric và mô hình.

<a id="s5"></a>
## 5. Phân tích khám phá (EDA)

Mỗi mục con trả lời một câu hỏi. Các góc nhìn bắt buộc đi trước (phân phối, quan hệ với target, tương quan); từ mục 5.6 là các **phân tích theo câu hỏi**: hình trả lời một câu hỏi, tiêu đề nêu kết luận và có số để so sánh.

### 5.1 Các đặc trưng số phân bố thế nào?

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:12]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(hist_bar(raw[c], bins=40), row=r, col=k)
fig.update_layout(showlegend=False)
show_fig(fig, "Distribution of the numeric features", height=280 * nrows)
raw[NUM_COLS].skew().sort_values(key=abs, ascending=False).rename("độ lệch (skewness)").to_frame()  # |skew| > 1: lệch mạnh

**Nhận xét.** ✏️ Hình dạng (đối xứng, lệch, hai đỉnh), các đỉnh nhọn ở giá trị đặc biệt (0, 99, …), số liệu skewness.

### 5.2 Category nào chiếm ưu thế ở mỗi đặc trưng phân loại?

In [ ]:
TOP_K = 10
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    vc = raw[c].value_counts(dropna=False)
    top = vc.head(TOP_K)
    if len(vc) > TOP_K:  # gộp phần đuôi dài để panel dễ đọc
        top = pd.concat([top, pd.Series({"(other)": vc.iloc[TOP_K:].sum()})])
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=top.index.astype(str), y=top.values, marker_color=BLUE, showlegend=False,
                         hovertemplate="%{x}: %{y:,}<extra></extra>"), row=r, col=k)
show_fig(fig, f"Top {TOP_K} categories per categorical feature", height=320 * nrows)

**Nhận xét.** ✏️ Category áp đảo và category hiếm, cardinality cao, giá trị cần gộp.

### 5.3 Đặc trưng số nào phân biệt được các lớp?

Boxplot theo lớp: các hộp càng ít chồng lên nhau, đặc trưng càng hữu ích. Bảng định lượng bằng median theo từng lớp.

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:9]
classes = raw[TARGET].dropna().unique()
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    for j, cls in enumerate(classes):
        fig.add_trace(go.Box(y=raw.loc[raw[TARGET] == cls, c], name=str(cls), marker_color=QUALITATIVE[j % 6],
                             boxpoints=False, showlegend=(i == 0), legendgroup=str(cls)), row=r, col=k)
show_fig(fig, f"Numeric features by '{TARGET}' class", height=300 * nrows)
raw.groupby(TARGET)[NUM_COLS].median().T  # median theo lớp: chênh lệch lớn = đặc trưng phân biệt tốt

**Nhận xét.** ✏️ 2–3 đặc trưng có median chênh lệch nhiều nhất giữa các lớp, kèm số liệu.

### 5.4 Tỷ lệ lớp thiểu số thay đổi thế nào theo từng category?

Với mỗi đặc trưng phân loại: tỷ lệ lớp thiểu số trong từng category (các category nhiều mẫu nhất). Đường chấm là tỷ lệ chung; cột càng xa đường này thì category càng mang thông tin.

In [ ]:
overall_rate = (raw[TARGET] == MINORITY).mean() * 100
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    top_categories = raw[c].value_counts().head(10).index
    rate = (raw[raw[c].isin(top_categories)].groupby(c)[TARGET]
            .apply(lambda s: (s == MINORITY).mean() * 100).sort_values())
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=rate.index.astype(str), y=rate.values, marker_color=ORANGE, showlegend=False,
                         hovertemplate="%{x}: %{y:.1f}%<extra></extra>"), row=r, col=k)
    fig.add_hline(y=overall_rate, line_dash="dot", line_color=MUTED, row=r, col=k)
show_fig(fig, f"Share of '{MINORITY}' per category (dotted line = overall {overall_rate:.1f}%)", height=320 * nrows)

**Nhận xét.** ✏️ Category có tỷ lệ lớp thiểu số cao nhất / thấp nhất và cách tỷ lệ chung bao xa.

### 5.5 Các đặc trưng số có tương quan với nhau không?

Tương quan Pearson đo mối liên hệ **tuyến tính**. Cặp có |r| > 0,9 mang gần như cùng thông tin: mô hình tuyến tính không tách được ảnh hưởng của chúng, nên chỉ giữ một cột.

In [ ]:
corr = raw[NUM_COLS].corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=corr.round(2).values, texttemplate="%{text}", xgap=2, ygap=2,
                           colorbar=dict(title="r"), hovertemplate="%{y} x %{x}: %{z:.2f}<extra></extra>"))
show_fig(fig, "Pearson correlation between numeric features", height=max(420, 40 * len(NUM_COLS)),
         yaxis_autorange="reversed")

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()  # tam giác trên: mỗi cặp một lần
pairs[pairs.abs() > 0.9].rename("r").to_frame() if (pairs.abs() > 0.9).any() else print("Không có cặp nào có |r| > 0,9.")

**Nhận xét.** ✏️ Các cặp tương quan mạnh và cột nào bị bỏ (thêm vào `DROP_COLS` và giải thích).

### 5.6 Câu hỏi 1: ✏️ <câu hỏi>

Mỗi câu hỏi theo cùng mẫu: **câu hỏi → hình có tiêu đề kết luận → số liệu → kết luận**. Ví dụ dưới đây vẽ tỷ lệ lớp thiểu số theo hai đặc trưng cùng lúc (đặc trưng số được chia thành các nhóm theo phân vị).

In [ ]:
Q1_ROW, Q1_COL = "TODO", "TODO"  # TODO: tên hai đặc trưng


def as_groups(series, bins=5):
    """Cột số -> các nhóm theo phân vị (số mẫu bằng nhau); cột phân loại -> giữ nguyên."""
    return pd.qcut(series, q=bins, duplicates="drop").astype(str) if pd.api.types.is_numeric_dtype(series) else series


if "TODO" not in (Q1_ROW, Q1_COL):
    table = (raw.assign(_row=as_groups(raw[Q1_ROW]), _col=as_groups(raw[Q1_COL]), _hit=raw[TARGET] == MINORITY)
                .pivot_table(index="_row", columns="_col", values="_hit", aggfunc="mean") * 100)
    fig = go.Figure(go.Heatmap(z=table.values, x=table.columns.astype(str), y=table.index.astype(str),
                               colorscale=SEQ_BLUE, text=table.round(1).values, texttemplate="%{text}%",
                               colorbar=dict(title="%"), hovertemplate=f"{Q1_ROW}=%{{y}}, {Q1_COL}=%{{x}}: %{{z:.1f}}%<extra></extra>"))
    show_fig(fig, f"✏️ <conclusion>: share of '{MINORITY}' by {Q1_ROW} and {Q1_COL}",
             xaxis_title=Q1_COL, yaxis_title=Q1_ROW, height=460)

**Kết luận.** ✏️ Trả lời câu hỏi bằng số liệu từ hình.

### 5.7 Câu hỏi 2: ✏️ <câu hỏi>

In [ ]:
# TODO: phân tích cho câu hỏi 2 (tính toán -> hình bằng show_fig -> in các số dùng trong kết luận).

**Kết luận.** ✏️

<a id="s6"></a>
## 6. Chuẩn bị dữ liệu

Thứ tự theo bài giảng (module *Pandas*, "fit trên train, transform phần còn lại"):

| Khi nào | Làm gì | Vì sao |
|---|---|---|
| **Trước** khi chia, trên toàn bảng | strip chữ, chuẩn hóa chính tả, đổi chữ→số, bỏ dòng không có nhãn, bỏ dòng trùng hoàn toàn, bỏ giá trị vô lý | Quy tắc theo từng dòng: không dùng thống kê của dòng khác nên không gây rò rỉ. Bỏ dòng trùng trước giúp một mẫu không nằm ở cả train và test. |
| **Sau** khi chia, fit **chỉ trên train** | median / most frequent cho missing, mean & std để scale, danh sách category cho one-hot | Đây là thống kê của dữ liệu. Nếu fit trên toàn bảng, chúng "nhìn thấy" validation và test (leakage). |

### 6.1 Quy tắc làm sạch (trước khi chia)

In [ ]:
def normalize_values(df):
    """Sửa chữ theo từng dòng, dùng chung cho dữ liệu huấn luyện và dữ liệu mới (mục 9)."""
    out = df.copy()
    for c in out.columns.difference(out.select_dtypes("number").columns):
        s = out[c]
        s = s.where(s.isna(), s.astype(str).str.strip())   # chỉ strip giá trị thật; NaN giữ nguyên
        out[c] = s.replace(NA_TOKENS, np.nan)              # chuỗi missing ẩn -> NaN
    for c, mapping in SPELLING_MAPS.items():
        lowered = out[c].str.lower()
        out[c] = lowered.map(mapping).fillna(out[c])       # biến thể có trong map -> dạng chuẩn; còn lại giữ nguyên
    for c in TO_NUMERIC:
        out[c] = pd.to_numeric(out[c].astype(str).str.replace(",", ""), errors="coerce")
    return out


def clean_table(df):
    """Áp mọi quy tắc an toàn trước khi chia và ghi lại số dòng sau mỗi bước."""
    steps = [("dữ liệu gốc", len(df))]
    out = normalize_values(df)
    out = out.drop(columns=DROP_COLS)
    out = out.dropna(subset=[TARGET])                      # dòng không có nhãn không dùng được để huấn luyện hay kiểm tra
    steps.append(("có nhãn", len(out)))
    out = out.drop_duplicates()
    steps.append(("bỏ dòng trùng hoàn toàn", len(out)))
    valid = pd.Series(True, index=out.index)
    for c, (low, high) in VALID_RANGES.items():
        ok = out[c].isna()                                 # giá trị thiếu không phải vô lý: imputer sẽ xử lý
        ok |= out[c].between(-np.inf if low is None else low, np.inf if high is None else high)
        valid &= ok
    out = out.loc[valid]
    steps.append(("bỏ giá trị vô lý", len(out)))
    return out.reset_index(drop=True), pd.DataFrame(steps, columns=["bước", "số dòng"])


df, clean_steps = clean_table(raw)
FEATURES, NUM_COLS, CAT_COLS = column_roles(df)  # kiểu dữ liệu có thể đã đổi (TO_NUMERIC), nên tính lại vai trò cột
print("Numeric:", NUM_COLS)
print("Categorical:", CAT_COLS)
clean_steps.assign(**{"bị bỏ": lambda t: t["số dòng"].shift(1) - t["số dòng"]})

**Nhận xét.** ✏️ Số dòng bị bỏ ở từng bước và lý do; giải thích từng cột trong `DROP_COLS`.

### 6.2 Chia train / validation / test

- **Train (70%)** để fit tiền xử lý và mô hình; **validation (15%)** để so sánh và tinh chỉnh mô hình; **test (15%)** chỉ dùng **một lần** ở mục 8 để báo cáo điểm cuối cùng.
- `stratify` giữ tỷ lệ các lớp như nhau ở ba phần; `random_state=SEED` giúp chia lại được y hệt.
- Nhãn được mã hóa thành số nguyên 0, 1, … (`LabelEncoder`) để mọi metric hoạt động giống nhau cho bài toán nhị phân và nhiều lớp.

In [ ]:
X = df[FEATURES]
MINORITY = df[TARGET].value_counts().idxmin()      # tính lại: làm sạch có thể đã đổi cách viết nhãn
label_encoder = LabelEncoder().fit(df[TARGET])
y = label_encoder.transform(df[TARGET])            # tên lớp -> 0..K-1, theo thứ tự bảng chữ cái
CLASS_NAMES = [str(c) for c in label_encoder.classes_]
POS = CLASS_NAMES.index(str(MINORITY))             # chỉ số của lớp thiểu số (lớp "dương" trong bài toán nhị phân)
IS_BINARY = len(CLASS_NAMES) == 2

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE + VAL_SIZE, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE), stratify=y_temp, random_state=SEED)

split_table = pd.DataFrame({
    "tập": ["train", "validation", "test"],
    "số dòng": [len(X_train), len(X_val), len(X_test)],
    "tỷ lệ": [len(p) / len(X) for p in (X_train, X_val, X_test)],
    f"% '{MINORITY}'": [np.mean(p == POS) * 100 for p in (y_train, y_val, y_test)],
})
split_table

### 6.3 Encoding và scaling, chỉ fit trên train

| Loại cột | Bước 1 | Bước 2 |
|---|---|---|
| numeric | `SimpleImputer(strategy="median")`: median không nhạy với outlier | `StandardScaler`: $z = (x - \mu_{train}) / \sigma_{train}$, cần cho mô hình tuyến tính; mô hình cây không bị ảnh hưởng |
| categorical | `SimpleImputer(strategy="most_frequent")` | `OneHotEncoder`: mỗi category một cột 0/1. Nếu mã hóa thành số nguyên, mô hình sẽ hiểu category có thứ tự lớn nhỏ. `handle_unknown="infrequent_if_exist"` đưa category chưa gặp ở train vào cột infrequent thay vì báo lỗi; `min_frequency` gộp các category hiếm. |

`ColumnTransformer` đưa mỗi nhóm cột vào pipeline của nó. `preprocessor` **chưa fit** được đặt trong pipeline của từng mô hình ở mục 7, nên `fit` chỉ học thống kê từ các dòng huấn luyện, kể cả trong cross-validation.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=MIN_CATEGORY_FREQ,
                              sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUM_COLS),
    ("cat", categorical_pipeline, CAT_COLS),
])

# Fit một lần ở đây chỉ để kiểm tra kết quả; các mô hình bên dưới tự fit bản sao của riêng chúng.
prep_check = clone(preprocessor).fit(X_train)                 # học median, mode, mean, std, danh sách category
X_train_ml = prep_check.transform(X_train)
X_val_ml, X_test_ml = prep_check.transform(X_val), prep_check.transform(X_test)
feature_names = prep_check.get_feature_names_out()

print(f"train {X_train_ml.shape} | validation {X_val_ml.shape} | test {X_test_ml.shape}")
print(f"{len(NUM_COLS)} cột numeric + {len(CAT_COLS)} cột categorical -> {len(feature_names)} cột cho mô hình")
assert not np.isnan(X_train_ml).any() and not np.isnan(X_test_ml).any(), "Còn NaN sau tiền xử lý"
pd.Series(prep_check.named_transformers_["num"].named_steps["imputer"].statistics_, index=NUM_COLS,
          name="median học từ train").to_frame()

### 6.4 Trước / sau khi scale

Scaling thay đổi **thang đo** (mean 0, độ lệch chuẩn 1), không thay đổi **hình dạng** phân phối. Hình kiểm tra điều đó trên hai cột số có độ phân tán lớn nhất.

In [ ]:
show_cols = X_train[NUM_COLS].std().sort_values(ascending=False).index[:2].tolist()
fig = make_subplots(rows=len(show_cols), cols=2, vertical_spacing=0.2,
                    subplot_titles=[t for c in show_cols for t in (f"{c}: before", f"{c}: after StandardScaler")])
for r, c in enumerate(show_cols, start=1):
    j = list(feature_names).index(f"num__{c}")
    fig.add_trace(hist_bar(X_train[c], color=BLUE), row=r, col=1)
    fig.add_trace(hist_bar(X_train_ml[:, j], color=ORANGE), row=r, col=2)
fig.update_layout(showlegend=False)
show_fig(fig, "StandardScaler changes the scale, not the shape of the distribution", height=300 * len(show_cols))

<a id="s7"></a>
## 7. Mô hình hóa

### 7.1 So sánh mô hình trên tập validation

Mỗi mô hình là một `Pipeline(preprocessor → model)`, nên tiền xử lý luôn được fit trên đúng dữ liệu mà mô hình được fit.

| Mô hình | Vì sao có trong so sánh |
|---|---|
| `DummyClassifier` | **Baseline**: luôn đoán lớp đa số. Mô hình hữu ích phải vượt rõ baseline. |
| `LogisticRegression` | Tuyến tính, nhanh, hệ số dễ giải thích. |
| `DecisionTreeClassifier` | Luật phi tuyến, dễ giải thích; `max_depth` hạn chế overfitting. |
| `RandomForestClassifier` | Trung bình của nhiều cây: ổn định hơn một cây đơn. |
| `HistGradientBoostingClassifier` | Các cây được xây nối tiếp, cây sau sửa lỗi cây trước; mạnh trên dữ liệu bảng. |
| Các biến thể `class_weight="balanced"` | Tăng trọng số lỗi của lớp thiểu số; so với bản thường để đo hiệu quả của việc xử lý mất cân bằng. |

Metric chính là **F1-macro** (trung bình F1 của các lớp, mỗi lớp có trọng số như nhau, nên chịu được mất cân bằng). Accuracy, balanced accuracy và ROC-AUC được báo cáo kèm.

In [ ]:
def evaluate(model, X, y):
    """Các metric phân loại của một mô hình đã fit trên một tập dữ liệu."""
    pred = model.predict(X)
    scores = {"accuracy": accuracy_score(y, pred),
              "balanced_accuracy": balanced_accuracy_score(y, pred),
              "f1_macro": f1_score(y, pred, average="macro")}
    proba = model.predict_proba(X)
    scores["roc_auc"] = (roc_auc_score(y == POS, proba[:, POS]) if IS_BINARY
                         else roc_auc_score(y, proba, multi_class="ovr"))
    return scores


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                       n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEED),
}

fitted, rows = {}, []
for name, estimator in MODELS.items():
    model = make_model(estimator).fit(X_train, y_train)
    fitted[name] = model
    rows.append({"model": name, **evaluate(model, X_val, y_val)})
    print(f"xong: {name}")

results = pd.DataFrame(rows).set_index("model").sort_values("f1_macro", ascending=False)
results

In [ ]:
plot_df = results.reset_index().melt(id_vars="model", var_name="metric", value_name="score")
fig = px.bar(plot_df, x="score", y="model", color="metric", orientation="h", barmode="group",
             category_orders={"model": results.index.tolist()[::-1]})
fig.update_xaxes(range=[0, 1])
show_fig(fig, f"Validation scores: best F1-macro = {results['f1_macro'].max():.3f} ({results['f1_macro'].idxmax()})",
         height=500, xaxis_title="Score on validation", yaxis_title="")

**Nhận xét.** ✏️ Xếp hạng mô hình; khoảng cách với baseline; hiệu quả của `class_weight="balanced"` (bản thường vs bản balanced).

### 7.2 Tinh chỉnh siêu tham số

`GridSearchCV` thử mọi tổ hợp trong `PARAM_GRID` bằng **cross-validation k-fold phân tầng trên tập train** (validation và test không bị đụng tới). Tên tham số có tiền tố `model__` vì estimator là bước `"model"` của pipeline.

In [ ]:
TUNE_MODEL = "Random Forest (balanced)"  # TODO: thường là họ mô hình tốt nhất ở 7.1
PARAM_GRID = {                           # TODO: sửa theo TUNE_MODEL (tên tham số khác nhau giữa các mô hình)
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 5],
}

search = GridSearchCV(make_model(MODELS[TUNE_MODEL]), PARAM_GRID, scoring="f1_macro",
                      cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED), n_jobs=-1)
search.fit(X_train, y_train)

tuned_name = f"{TUNE_MODEL} (tuned)"
fitted[tuned_name] = search.best_estimator_
results.loc[tuned_name] = pd.Series(evaluate(search.best_estimator_, X_val, y_val))  # dòng mới, khớp theo tên cột
print("Tham số tốt nhất:", search.best_params_, f"| CV F1-macro = {search.best_score_:.3f}")
(pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
   .sort_values("rank_test_score").head(10))

### 7.3 Chọn mô hình cuối cùng

Mô hình có F1-macro tốt nhất trên **validation** được fit lại trên train + validation (nhiều dữ liệu hơn, siêu tham số giữ nguyên) và được đánh giá một lần trên test ở mục 8.

In [ ]:
results = results.sort_values("f1_macro", ascending=False)
BEST_NAME = results.index[0]
final_model = clone(fitted[BEST_NAME]).fit(pd.concat([X_train, X_val]), np.concatenate([y_train, y_val]))
baseline_model = fitted["Baseline (majority class)"]
print(f"Mô hình cuối: {BEST_NAME} (F1-macro trên validation {results.loc[BEST_NAME, 'f1_macro']:.3f})")
results

**Nhận xét.** ✏️ Mô hình được chọn, lý do, và mức cải thiện so với baseline.

<a id="s8"></a>
## 8. Đánh giá trên tập test

Tập test chưa được dùng cho bất kỳ quyết định nào, nên các số liệu này ước lượng hiệu năng trên dữ liệu mới.

### 8.1 Metric trên test

In [ ]:
test_scores = pd.DataFrame({"Baseline (majority class)": evaluate(baseline_model, X_test, y_test),
                            f"Final: {BEST_NAME}": evaluate(final_model, X_test, y_test)}).T
y_pred = final_model.predict(X_test)
display(test_scores)
pd.DataFrame(classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)).T

### 8.2 Confusion matrix

Hàng = lớp thật, cột = lớp dự đoán. Mỗi ô ghi số lượng và tỷ lệ trên lớp thật (% theo hàng), cho thấy recall của từng lớp.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_row = cm / cm.sum(axis=1, keepdims=True) * 100
fig = go.Figure(go.Heatmap(z=cm_row, x=CLASS_NAMES, y=CLASS_NAMES, colorscale=SEQ_BLUE, zmin=0, zmax=100,
                           text=[[f"{n:,}<br>({p:.1f}%)" for n, p in zip(rn, rp)] for rn, rp in zip(cm, cm_row)],
                           texttemplate="%{text}", colorbar=dict(title="row %"),
                           hovertemplate="true %{y}, predicted %{x}: %{z:.1f}%<extra></extra>"))
show_fig(fig, f"Confusion matrix on test: recall of '{MINORITY}' = {cm_row[POS, POS]:.1f}%",
         xaxis_title="Predicted class", yaxis_title="True class", yaxis_autorange="reversed", height=460)

**Nhận xét.** ✏️ Lớp nào bị nhầm với lớp nào và bao nhiêu lần (số liệu từ ma trận).

### 8.3 Đường ROC và precision–recall (bài toán nhị phân)

Đường ROC cho thấy đánh đổi giữa tỷ lệ dương thật và dương giả trên mọi ngưỡng; khi các lớp mất cân bằng, đường precision–recall cho nhiều thông tin hơn.

In [ ]:
if IS_BINARY:
    proba_pos = final_model.predict_proba(X_test)[:, POS]
    fpr, tpr, _ = roc_curve(y_test == POS, proba_pos)
    precision, recall, _ = precision_recall_curve(y_test == POS, proba_pos)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"ROC (AUC = {roc_auc_score(y_test == POS, proba_pos):.3f})",
        f"Precision–recall (AP = {average_precision_score(y_test == POS, proba_pos):.3f})"))
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(color=BLUE, width=2), name="ROC"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=MUTED, dash="dot"), name="random"), row=1, col=1)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", line=dict(color=ORANGE, width=2), name="PR"), row=1, col=2)
    fig.add_hline(y=np.mean(y_test == POS), line_dash="dot", line_color=MUTED, row=1, col=2)
    fig.update_xaxes(title_text="False positive rate", row=1, col=1)
    fig.update_yaxes(title_text="True positive rate", row=1, col=1)
    fig.update_xaxes(title_text="Recall", row=1, col=2)
    fig.update_yaxes(title_text="Precision", row=1, col=2)
    show_fig(fig, f"Threshold-free view of the final model on test (positive class = '{MINORITY}')", height=440)
else:
    print("Target nhiều lớp: xem báo cáo theo từng lớp ở mục 8.1.")

### 8.4 Đặc trưng nào quan trọng?

**Permutation importance**: xáo trộn lần lượt từng cột gốc trên tập test và đo F1-macro giảm bao nhiêu. Cách này dùng được cho mọi mô hình và tính trên các cột gốc (trước one-hot).

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="f1_macro", n_repeats=5,
                              random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"mức giảm F1-macro trung bình": perm.importances_mean, "độ lệch chuẩn": perm.importances_std},
                           index=X_test.columns).sort_values("mức giảm F1-macro trung bình").tail(15))
fig = go.Figure(go.Bar(x=importance["mức giảm F1-macro trung bình"], y=importance.index, orientation="h",
                       error_x=dict(type="data", array=importance["độ lệch chuẩn"], color=MUTED), marker_color=BLUE,
                       hovertemplate="%{y}: %{x:.4f}<extra></extra>"))
show_fig(fig, f"Permutation importance on test: '{importance.index[-1]}' matters most",
         xaxis_title="Drop in F1-macro when the column is shuffled", height=max(380, 30 * len(importance)))

**Nhận xét.** ✏️ Các đặc trưng quan trọng nhất; có khớp với EDA ở mục 5 không?

### 8.5 Phân tích lỗi

Các dòng test bị dự đoán sai, sai mà tự tin nhất xếp trước. ✏️ Tìm quy luật (một nhóm con, missing values, outlier).

In [ ]:
proba = final_model.predict_proba(X_test)
errors = X_test.assign(true=label_encoder.inverse_transform(y_test),
                       predicted=label_encoder.inverse_transform(y_pred),
                       confidence=proba.max(axis=1))
errors = errors[errors["true"] != errors["predicted"]].sort_values("confidence", ascending=False)
print(f"{len(errors):,} dự đoán sai trên {len(X_test):,} dòng test ({len(errors) / len(X_test):.1%})")
errors.head(10)

**Nhận xét.** ✏️

<a id="s9"></a>
## 9. Dự đoán trên dữ liệu mới

Dùng pipeline cuối từ đầu đến cuối: dòng dữ liệu mới chưa xử lý → cùng các bước sửa chữ ở 6.1 (`normalize_values`) → tiền xử lý → dự đoán. Cột bị thiếu trong dòng mới thành `NaN` và được điền giá trị giống hệt lúc huấn luyện.

In [ ]:
# TODO: thay bằng các dòng tự viết, vd pd.DataFrame([{"age": 35, "city": " hcm", ...}])
new_rows = X_test.head(3).copy()

new_rows = normalize_values(new_rows).reindex(columns=FEATURES)
pred_proba = final_model.predict_proba(new_rows)
prediction = new_rows.assign(
    predicted=label_encoder.inverse_transform(pred_proba.argmax(axis=1)),
    **{f"P({c})": pred_proba[:, i] for i, c in enumerate(CLASS_NAMES)})
prediction

<a id="s10"></a>
## 10. Tổng kết

### 10.1 Tóm tắt kết quả

| Mô hình | Accuracy | Balanced accuracy | F1-macro | ROC-AUC |
|---|---|---|---|---|
| Baseline (majority class) | ✏️ | ✏️ | ✏️ | ✏️ |
| ✏️ mô hình cuối | ✏️ | ✏️ | ✏️ | ✏️ |

### 10.2 Những gì phân tích đã trả lời

✏️ 3–5 phát hiện, mỗi phát hiện có một con số từ notebook và dẫn tới hình tương ứng (vd "Figure 7 cho thấy …").

### 10.3 Hạn chế

✏️ Dữ liệu (kích thước, giai đoạn, sai lệch lấy mẫu, nhiễu nhãn), phương pháp (giả định, đặc trưng chưa dùng), đánh giá (một lần chia, phân phối của tập test).

### 10.4 Hướng mở rộng

✏️ Những gì sẽ thử tiếp theo.